# Tournament Results Analysis

This notebook loads a saved tournament `results.csv`, summarizes the run, and plots a few balance-oriented views.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
CSV_PATH = None  # Optional: set this to a specific results.csv Path(...) if desired.


def find_latest_results_csv(root: Path) -> Path:
    candidates = []
    search_roots = [
        root / 'artifacts' / 'tournaments',
        root / 'tests' / '_artifacts_tmp',
    ]

    for search_root in search_roots:
        if search_root.exists():
            candidates.extend(search_root.rglob('results.csv'))

    if not candidates:
        raise FileNotFoundError('No results.csv file was found under artifacts/tournaments or tests/_artifacts_tmp.')

    return max(candidates, key=lambda path: path.stat().st_mtime)


results_path = Path(CSV_PATH) if CSV_PATH else find_latest_results_csv(ROOT)
results_path

In [ ]:
df = pd.read_csv(results_path)
df.head()

In [ ]:
summary = {
    'games': len(df),
    'p1_winrate': (df['winner'] == 1).mean(),
    'p2_winrate': (df['winner'] == 2).mean(),
    'draw_rate': df['winner'].isna().mean(),
    'avg_turns': df['turns'].mean(),
    'avg_rounds': df['rounds'].mean(),
    'avg_temperature': df['temperature'].mean(),
    'avg_ph': df['ph'].mean(),
    'avg_current_era': df['current_era'].mean(),
    'avg_p1_score': df['p1_score'].mean(),
    'avg_p2_score': df['p2_score'].mean(),
    'avg_board_occupancy': df['board_occupancy'].mean(),
}

pd.Series(summary).to_frame('value')

In [ ]:
df.describe(include='all').T

## Winner Distribution

In [ ]:
winner_counts = df['winner'].fillna('draw').astype(str).value_counts().sort_index()

ax = winner_counts.plot(kind='bar', figsize=(6, 4), title='Winner Distribution')
ax.set_xlabel('Winner')
ax.set_ylabel('Games')
plt.tight_layout()
plt.show()

## Score Comparison

In [ ]:
ax = df[['p1_score', 'p2_score']].plot(kind='hist', bins=15, alpha=0.65, figsize=(8, 4), title='Score Distribution')
ax.set_xlabel('Score')
plt.tight_layout()
plt.show()

## Climate Outcomes

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df['temperature'].plot(kind='hist', bins=12, ax=axes[0], title='Final Temperature')
axes[0].set_xlabel('Temperature')

df['ph'].plot(kind='hist', bins=12, ax=axes[1], title='Final pH')
axes[1].set_xlabel('pH')

df['current_era'].value_counts().sort_index().plot(kind='bar', ax=axes[2], title='Final Era')
axes[2].set_xlabel('Era')
axes[2].set_ylabel('Games')

plt.tight_layout()
plt.show()

## Efficiency and Pace

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df[['p1_efficiency', 'p2_efficiency']].plot(kind='box', ax=axes[0], title='Efficiency by Player')
axes[0].set_ylabel('Score / spent resources')

df[['turns', 'rounds']].plot(kind='hist', bins=12, alpha=0.7, ax=axes[1], title='Game Length')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.show()

## Correlations

In [ ]:
numeric_cols = [
    'turns', 'rounds', 'temperature', 'ph', 'current_era',
    'consumed_climate_cards', 'remaining_climate_cards',
    'board_occupancy', 'p1_score', 'p2_score',
    'p1_efficiency', 'p2_efficiency',
]

corr = df[numeric_cols].corr(numeric_only=True)
corr.style.background_gradient(cmap='coolwarm').format(precision=2)